# 📊 GIAI ĐOẠN 1: MACHINE LEARNING CHO DỮ LIỆU BẢNG (TABULAR DATA)
Notebook này thực hành quy trình chuẩn xây dựng Pipeline Machine Learning cho kỳ thi OlpAI:
1. **Feature Engineering**: Handling Missing Values, One-Hot Encoding, StandardScaler.
2. **Stratified K-Fold Cross Validation**: Đánh giá nếp gấp chống Overfitting.
3. **Tree-based Models**: XGBoost, LightGBM, CatBoost.
4. **Ensembling and Blending**: Đánh giá Out-Of-Fold (OOF) và xuất kết quả `submission.csv`.

In [1]:
import numpy as np
import pandas as pd
import random
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# 1. Cố định Seed
def seed_everything(seed=2026):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(2026)
print("✅ Đã thiết lập Seed 2026 thành công!")

✅ Đã thiết lập Seed 2026 thành công!


In [2]:
print("--- 🎯 BÀI TẬP 1: PREPROCESSING VÀ FEATURE ENGINEERING ---")

# Tạo 500 mẫu dữ liệu giả lập có cả số và thuộc tính phân loại
n_samples = 500
data = {
    'num_feat_1': np.random.randn(n_samples),
    'num_feat_2': np.random.randn(n_samples) * 10,
    'cat_feat_1': np.random.choice(['Category_A', 'Category_B', 'Category_C'], size=n_samples),
    'target': np.random.randint(0, 2, size=n_samples)
}
df = pd.DataFrame(data)

# Tạo giá trị thiếu (NaN) giả lập
nan_mask = np.random.choice([True, False], size=n_samples, p=[0.1, 0.9])
df.loc[nan_mask, 'num_feat_1'] = np.nan

# Xử lý thiếu dữ liệu (Imputation)
df['num_feat_1'] = df['num_feat_1'].fillna(df['num_feat_1'].median())

X = df.drop(columns=['target'])
y = df['target']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['num_feat_1', 'num_feat_2']),
        ('cat', OneHotEncoder(sparse_output=False), ['cat_feat_1'])
    ]
)

X_processed = np.array(preprocessor.fit_transform(X))
print(f"Shape dữ liệu sau khi tiền xử lý: {X_processed.shape}")

--- 🎯 BÀI TẬP 1: PREPROCESSING VÀ FEATURE ENGINEERING ---
Shape dữ liệu sau khi tiền xử lý: (500, 5)


In [4]:
print("--- 🌲 BÀI TẬP 2: STRATIFIED K-FOLD CV VỚI XGBOOST VÀ LIGHTGBM ---")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
oof_preds_xgb = np.zeros(len(df))
oof_preds_lgb = np.zeros(len(df))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_processed, y)):
    X_tr, y_tr = X_processed[train_idx], y.iloc[train_idx]
    X_va, y_va = X_processed[val_idx], y.iloc[val_idx]
    
    # Huấn luyện XGBoost
    model_xgb = xgb.XGBClassifier(n_estimators=50, random_state=2026, eval_metric='logloss')
    model_xgb.fit(X_tr, y_tr)
    probs_xgb = np.array(model_xgb.predict_proba(X_va))
    oof_preds_xgb[val_idx] = probs_xgb[:, 1]
    
    # Huấn luyện LightGBM
    model_lgb = lgb.LGBMClassifier(n_estimators=50, random_state=2026, verbose=-1)
    model_lgb.fit(X_tr, y_tr)
    probs_lgb = np.array(model_lgb.predict_proba(X_va))
    oof_preds_lgb[val_idx] = probs_lgb[:, 1]

# Đánh giá điểm Out-Of-Fold (OOF)
acc_xgb = accuracy_score(y, (oof_preds_xgb >= 0.5).astype(int))
acc_lgb = accuracy_score(y, (oof_preds_lgb >= 0.5).astype(int))
acc_blend = accuracy_score(y, ((oof_preds_xgb + oof_preds_lgb)/2 >= 0.5).astype(int))

print(f"OOF Accuracy XGBoost: {acc_xgb:.4f}")
print(f"OOF Accuracy LightGBM: {acc_lgb:.4f}")
print(f"OOF Accuracy Ensemble: {acc_blend:.4f}")

--- 🌲 BÀI TẬP 2: STRATIFIED K-FOLD CV VỚI XGBOOST VÀ LIGHTGBM ---
OOF Accuracy XGBoost: 0.4880
OOF Accuracy LightGBM: 0.4900
OOF Accuracy Ensemble: 0.4920
